# AI4I Exploration
Quick EDA for machine failure patterns in AI4I.

In [5]:
from pathlib import Path
import pandas as pd

path = Path('../data/ai4i2020.csv')
if not path.exists():
    path = Path('../data/raw/ai4i2020.csv')

df = pd.read_csv(path)
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [6]:
df.shape, df.columns.tolist()

((10000, 14),
 ['UDI',
  'Product ID',
  'Type',
  'Air temperature [K]',
  'Process temperature [K]',
  'Rotational speed [rpm]',
  'Torque [Nm]',
  'Tool wear [min]',
  'Machine failure',
  'TWF',
  'HDF',
  'PWF',
  'OSF',
  'RNF'])

In [7]:
failure_cols = ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df[failure_cols].mean().sort_values(ascending=False)

Machine failure    0.0339
HDF                0.0115
OSF                0.0098
PWF                0.0095
TWF                0.0046
RNF                0.0019
dtype: float64

In [8]:
(df.groupby('Type')['Machine failure'].mean() * 100).round(2)

Type
H    2.09
L    3.92
M    2.77
Name: Machine failure, dtype: float64

## Next Steps: Deeper Reliability EDA
Use these cells to check data quality, create a couple of interpretable features, and identify operating regions with elevated failure risk.

In [9]:
# 1) Data quality and label balance
missing = df.isna().sum().sort_values(ascending=False)
label_balance = df['Machine failure'].value_counts(normalize=True).rename('pct') * 100

missing.head(10), label_balance.round(2)

(UDI                        0
 Product ID                 0
 Type                       0
 Air temperature [K]        0
 Process temperature [K]    0
 Rotational speed [rpm]     0
 Torque [Nm]                0
 Tool wear [min]            0
 Machine failure            0
 TWF                        0
 dtype: int64,
 Machine failure
 0    96.61
 1     3.39
 Name: pct, dtype: float64)

In [10]:
# 2) Feature engineering for operating stress signals
work = df.copy()
work['delta_temp_k'] = work['Process temperature [K]'] - work['Air temperature [K]']
work['power_proxy'] = work['Torque [Nm]'] * work['Rotational speed [rpm]']

work[['delta_temp_k', 'power_proxy']].describe().T

,count,mean,std,min,25%,50%,75%,max
delta_temp_k,10000.0,10.00063,1.001094,7.6,9.3,9.8,11.00,12.1
power_proxy,10000.0,59967.14704,10193.093881,10966.8,53105.4,59883.9,66873.75,99980.4


In [11]:
# 3) Failure rate by stress bins (helps define inspection thresholds)
work['torque_bin'] = pd.qcut(work['Torque [Nm]'], q=5, duplicates='drop')
work['wear_bin'] = pd.qcut(work['Tool wear [min]'], q=5, duplicates='drop')

torque_failure = (work.groupby('torque_bin', observed=False)['Machine failure'].mean() * 100).round(2)
wear_failure = (work.groupby('wear_bin', observed=False)['Machine failure'].mean() * 100).round(2)

torque_failure, wear_failure

(torque_bin
 (3.799, 31.4]     2.35
 (31.4, 37.5]      0.45
 (37.5, 42.7]      0.59
 (42.7, 48.3]      2.17
 (48.3, 76.6]     11.50
 Name: Machine failure, dtype: float64,
 wear_bin
 (-0.001, 42.0]    2.37
 (42.0, 86.0]      2.09
 (86.0, 130.0]     2.33
 (130.0, 174.0]    2.15
 (174.0, 253.0]    8.14
 Name: Machine failure, dtype: float64)

In [12]:
# 4) Build a shortlist of high-risk observations for review
top_risk = work.sort_values(
    by=['Machine failure', 'Tool wear [min]', 'Torque [Nm]', 'Rotational speed [rpm]'],
    ascending=[False, False, False, False]
 )

top_risk[['UDI', 'Product ID', 'Type', 'Machine failure', 'Tool wear [min]', 'Torque [Nm]', 'Rotational speed [rpm]', 'delta_temp_k']].head(20)

,UDI,Product ID,Type,Machine failure,Tool wear [min],Torque [Nm],Rotational speed [rpm],delta_temp_k
5401,5402,M20261,M,1,253,54.8,1454,9.7
5400,5401,L52580,L,1,251,46.3,1477,9.7
5399,5400,H34813,H,1,246,53.8,1411,9.6
2864,2865,H32278,H,1,246,47.6,1380,8.8
9667,9668,L56847,L,1,238,48.9,1352,11.1
4034,4035,L51214,L,1,235,29.0,1615,8.8
5394,5395,M20254,M,1,234,70.5,1262,9.5
4646,4647,L51826,L,1,234,46.0,1497,8.1
5999,6000,M20859,M,1,234,30.5,1671,9.8
4385,4386,L51565,L,1,233,45.3,1442,7.9
